# Four-day 250-stock ablation

The unattended goal runner executes the complete manifest. This notebook inspects progress, diagnostics, forecasts and inference without repeating training. Use the **Hypershift CUDA experiment** kernel.

35 core configurations × four intervals × three folds × three seeds = 1,260 fits. The 48 latest-fold contrasts add 384 fits. Twelve joint tuning trials per interval use only first-fold history. PH localization, H1/H2 context, Cover Learning and Gromov diagnostics run before forecasting.

In [ ]:
from pathlib import Path
import sys, json
bases = [Path.cwd(), Path.cwd() / 'Hypershift', Path('C:\\Projects\\ACM Research\\Hypershift')]
ROOT = next(p for base in bases for p in [base, *base.parents] if (p / 'experiments/market_ablation/config.py').exists())
sys.path.insert(0, str(ROOT))
from experiments.market_ablation.config import SweepConfig, manifest, core_variants, contrast_variants
from experiments.market_ablation.storage import read_json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
CONFIG = SweepConfig()
OUT = CONFIG.root
print('Output:', OUT)
print('Final fits:', manifest(CONFIG)['final_fits'])

In [ ]:
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print('Progress:', read_json(OUT / 'progress.json'))
print('Preparation:', read_json(OUT / 'dataset/preparation.json', {}).get('complete', False))
preflight = read_json(OUT / 'preflight.json') or read_json(OUT / 'selected_preflight.json') or read_json(OUT / 'early_preflight.json', {})
print('Preflight:', {k:preflight.get(k) for k in ['passed','projected_hours','target_hours','exceeds_target','scope']})
print('Finished final fits:', len(list((OUT / 'fits').glob('*/fold-[012]/*/seed-*/result.json'))))

In [ ]:
display(pd.DataFrame([{'variant': v.name, 'families': ''.join(v.families) or 'temporal', 'features': v.features, 'PH_context': v.ph_context, 'control': v.control} for v in core_variants()+contrast_variants()]))
print('PH gate:', read_json(OUT / 'validation/ph_gate.json'))
print('Cover gate:', read_json(OUT / 'validation/cover_gate.json'))
print('Landmark gate:', read_json(OUT / 'validation/landmark_gate.json'))
from IPython.display import Image
cover_graph = OUT / 'validation/figures/1m-fold-0-cover.png'
if cover_graph.exists():
    display(Image(filename=str(cover_graph)))
constructor_inventory = read_json(OUT / 'validation/constructor-inventory/verification.json')
if constructor_inventory:
    print('Initial constructor recipes:', constructor_inventory['constructor_recipes'])
    print('Empty financial P base recipes:', constructor_inventory['empty_PH_base_recipes'], '/ 12')
    print('Empty P comparisons do not assess nonempty localized edges; PH context is evaluated separately.')
    print('Constructor tables/figure:', OUT / 'validation/constructor-inventory')


The archive retains every regular-session minute. Training rotates every fifth minute origin; the target is still the next minute. Validation/test support is common across feature packs. Final forecasts cover every eligible test origin. Static universe/GICS metadata are retrospective; portfolio economic accounting is kept separate from the raw forecasting target. Undefined ratios and unavailable corporate-action valuations are explicit.

In [ ]:
geometry = []
for p in (OUT / 'geometry').glob('*/monthly/*.json'):
    x = read_json(p)
    geometry.append({k:x.get(k) for k in ['month','interval','delta','relative_delta','diameter','coverage','missing_fraction']})
if geometry:
    gd = pd.DataFrame(geometry).sort_values('month')
    display(gd.tail(12))
    gd.pivot(index='month', columns='interval', values='relative_delta').plot(figsize=(12,4), title='Prepared stock-feature Gromov hyperbolicity')
    plt.ylabel('2δ / diameter')
else:
    print('Geometry is pending.')
structural = []
for p in (OUT / 'geometry/1m/fold-0').glob('*.json'):
    g = read_json(p)
    if 'projections' not in g:
        continue
    row = {'family':p.stem, 'edges':g['edges'], 'stock_coverage':g['covered_fraction']}
    for s, graph in g['projections'].items():
        largest = max(graph['components'], key=lambda c:c['points'])
        row[f'delta_hg_s{s}_largest_component'] = largest['delta']
    structural.append(row)
if structural:
    display(pd.DataFrame(structural).sort_values('family'))

In [ ]:
metrics_path = OUT / 'report/metrics.csv'
metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
if len(metrics):
    display(metrics.groupby(['interval','variant'])[['log_return_skill','price_skill','direction_accuracy','IC','rank_IC']].agg(['mean','std']))
else:
    print('Final statistics are pending; training results are available per completed fit.')

In [ ]:
# Change these selectors after fits have completed.
interval, fold, variant, seed, symbol = '1m', 2, 'A', 1001, 'MSFT'
job_dir = OUT / 'fits' / interval / f'fold-{fold}' / variant / f'seed-{seed}'
curve = read_json(job_dir / 'learning_curve.json', [])
if curve:
    display(pd.DataFrame(curve))
    pd.DataFrame(curve).plot(x='epoch', y='validation_persistence_relative_mse', marker='o', figsize=(8,3))
    plt.axhline(1, color='black', linestyle='--', label='persistence')
files = sorted((job_dir / 'predictions').glob('*.npz'))
if files:
    symbols = read_json(OUT / 'dataset/metadata.json')['symbols']
    col = symbols.index(symbol)
    with np.load(files[-1]) as p, np.load(OUT / 'shared' / interval / f'fold-{fold}' / 'test' / files[-1].name) as t:
        valid = t['mask'][:,col]
        selection = np.flatnonzero(valid)[-600:]
        plt.figure(figsize=(12,4))
        plt.plot(t['actual_price'][selection,col], label='actual')
        plt.plot(t['current_price'][selection,col]*np.exp(p['prediction'][selection,col]), label='model')
        plt.plot(t['current_price'][selection,col], label='naive persistence')
        plt.title(f'{symbol}: {interval} next-bar raw price'); plt.legend()

In [ ]:
analysis_dir = OUT / 'analysis' / interval / f'fold-{fold}' / variant / f'seed-{seed}'
portfolio = read_json(analysis_dir / 'portfolios/metrics.json')
if portfolio:
    rows = []
    for strategy, scenarios in portfolio['strategies'].items():
        for scenario, values in scenarios.items():
            rows.append({'strategy':strategy, 'scenario':scenario, **values})
    display(pd.DataFrame(rows)[['strategy','scenario','accounting_status','Sharpe','Sortino','IR','Calmar','MaxDD','VaR95','ES95','turnover']])
    for strategy in ['long_only','long_short']:
        with np.load(analysis_dir / 'portfolios' / f'{strategy}.npz') as p:
            fig, axes = plt.subplots(2,1,figsize=(12,5))
            axes[0].plot(p['wealth']); axes[0].set_title(strategy)
            axes[1].plot(p['drawdown']); axes[1].set_ylabel('Drawdown')
else:
    print('Portfolio statistics are pending.')

In [ ]:
comparisons = read_json(OUT / 'report/comparisons.json', [])
if comparisons:
    display(pd.DataFrame([{'interval':x['interval'],'fold':x['fold'],'variant':x['variant'],'scope':x['scope'],'paired_HAC_p':x['persistence_comparison'].get('primary_p'),'Holm_p':x.get('Holm_p_core_family')} for x in comparisons]))
primary = read_json(OUT / 'report/primary_relative_price_inference.json', {})
if primary:
    rows = []
    for x in primary['groups']:
        ci = x['price_skill_uncertainty'].get('HAC', {}).get(str(x['persistence_comparison'].get('primary_bandwidth')), {}).get('ci95')
        rows.append({'interval':x['interval'], 'fold':x['fold'], 'variant':x['variant'], 'scope':x['scope'],
                     'primary_price_skill_pct':100*x['price_skill'] if x['price_skill'] is not None else None,
                     'primary_HAC_ci95_pct':[100*v for v in ci] if ci else None,
                     'Holm_p_persistence':x.get('Holm_p_persistence'), 'reference':x.get('reference'),
                     'Holm_p_matched_reference':x.get('Holm_p_matched_reference')})
    primary_table = pd.DataFrame(rows)
    print('Primary score: equal-session relative price loss; market uncertainty is separate from seed variation.')
    display(primary_table[(primary_table.interval==interval)&(primary_table.fold==fold)].sort_values('primary_price_skill_pct', ascending=False))
else:
    print('Primary-score inference is pending the full manifest.')
pairs = read_json(OUT / 'report/feature_context_pair_inference.json', {}) or read_json(OUT / 'validation/feature-pair-interim' / interval / 'feature_context_pair_inference.json', {})
if pairs:
    selected = [x for x in pairs['groups'] if x['interval']==interval and x['anchor']==variant]
    print('Direct PH/feature comparisons: latest fold, two matching seeds; positive changes improve price skill.')
    display(pd.DataFrame([{'kind':x['kind'], 'candidate':x['candidate'], 'reference':x['reference'],
                          'skill_change_pp':100*x['skill_change_vs_matched_reference'],
                          'market_bootstrap_ci95_pp':[100*v for v in x['skill_change_uncertainty'].get('stationary_bootstrap', {}).get('10', {}).get('ci95', [])],
                          'Holm_p':x['Holm_p_matched_reference'], 'seed_change_std_pp':100*x['seed_skill_change_std']} for x in selected]))
else:
    print('Direct PH/feature inference is pending.')
print('SPA / Reality Check:', read_json(OUT / 'report/multiple_model_tests.json'))
print('Archive verification:', read_json(OUT / 'report/completion.json'))

All final float32 forecasts are saved losslessly by month, with shared targets/masks, selected checkpoints, optimizer/RNG resume states, constructor snapshots, PH diagrams, session contributions and portfolio paths. Further statistics can use these artifacts without retraining. To resume from a terminal: `.venv-experiment/Scripts/python.exe -u -m experiments.market_ablation run`. A process lock prevents two concurrent trainers.